# Coleta de Dados — Frota de Veículos Elétricos (SENATRAN)

Este notebook coleta os dados da frota de veículos elétricos no Brasil
a partir da SENATRAN via Base dos Dados (BigQuery).

**Fonte:** Secretaria Nacional de Trânsito (SENATRAN)  
**Acesso:** Base dos Dados (basedosdados.org)  
**Granularidade:** Por tipo de veículo, por UF, por ano  
**Período:** 2015 em diante

In [1]:
import basedosdados as bd
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Projeto Google Cloud
os.environ['GOOGLE_CLOUD_PROJECT'] = 'telemarketing-decline'

print("Bibliotecas carregadas!")

Bibliotecas carregadas!


In [2]:
# Verificar tabelas disponíveis do SENATRAN na Base dos Dados
query_catalogo = """
SELECT table_name
FROM `basedosdados.br_denatran_frota.INFORMATION_SCHEMA.TABLES`
"""

df_catalogo = bd.read_sql(
    query=query_catalogo,
    billing_project_id="telemarketing-decline"
)

print(df_catalogo)

BaseDosDadosAccessDeniedException: 
Are you sure you are using the right `billing_project_id`?
You must use the Project ID available in your Google Cloud console home page at https://console.cloud.google.com/home/dashboard
If you still don't have a Google Cloud Project, you have to create one.
You can set one up by following these steps: 
1. Go to this link https://console.cloud.google.com/projectselector2/home/dashboard
2. Agree with Terms of Service if asked
3. Click in Create Project
4. Put a cool name in your project
5. Hit create
6. Rerun this command with the flag `reauth=True`. 
   Like `read_table('br_ibge_pib', 'municipios', billing_project_id=<YOUR_PROJECT_ID>, reauth=True)`

In [3]:
query_senatran = """
SELECT
    ano,
    mes,
    sigla_uf,
    tipo,
    quantidade
FROM `basedosdados.br_denatran_frota.uf_tipo`
WHERE ano >= 2015
ORDER BY ano, mes, sigla_uf
"""

df_frota = bd.read_sql(
    query=query_senatran,
    billing_project_id="telemarketing-decline"
)

print(f"Dados carregados: {df_frota.shape}")
print(df_frota.head(10))

GenericGBQException: Reason: 400 POST https://bigquery.googleapis.com/bigquery/v2/projects/telemarketing-decline/queries?prettyPrint=false: Unrecognized name: tipo at [6:5]

In [27]:
query_catalogo = """
SELECT table_name
FROM `basedosdados.br_denatran_frota.INFORMATION_SCHEMA.TABLES`
"""

from google.cloud import bigquery

client = bigquery.Client(project="telemarketing-decline")
tables = client.list_tables("basedosdados.br_denatran_frota")

for table in tables:
    print(table.table_id)

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [5]:
import basedosdados as bd

df_frota = bd.read_sql(
    query="SELECT 1",
    billing_project_id="telemarketing-decline",
    reauth=True
)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform&state=dVTqA99gKD9WO4ThZ5hWS4Mv8jzRy4&code_challenge=Gq41qdR84DrwD--aVPf-BA1tHXNbw8aZd8AFEE_QnmY&code_challenge_method=S256&prompt=consent&access_type=offline
Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [7]:
from google.cloud import bigquery

client = bigquery.Client(project="telemarketing-decline")

dataset = client.get_dataset("basedosdados.br_denatran_frota")
tables = client.list_tables(dataset)

for table in tables:
    print(table.table_id)

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [28]:
import requests
import pandas as pd
from io import BytesIO

# URL base do portal SENATRAN
# Os arquivos são Excel por ano e mês
# Testar o acesso primeiro

url_teste = "https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/frota-de-veiculos-2024"

response = requests.get(url_teste)
print(f"Status: {response.status_code}")
print(f"Tamanho da resposta: {len(response.content)} bytes")

Status: 200
Tamanho da resposta: 177194 bytes


In [9]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.content, 'html.parser')

# Encontrar todos os links de arquivos
links = []
for a in soup.find_all('a', href=True):
    href = a['href']
    if any(ext in href.lower() for ext in ['.xlsx', '.xls', '.csv']):
        links.append({
            'texto': a.text.strip(),
            'url': href
        })

print(f"Links encontrados: {len(links)}")
for link in links:
    print(f"  {link['texto']} → {link['url']}")

Links encontrados: 112
  Frota por UF e Tipo de Veículo → https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/FrotaporUFeTipodeveiculoDezembro2024.xlsx
  Frota por Município e Tipo → https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/FrotapormunicipioetipoDezembro2024.xlsx
  Quantidade de Veículos por UF Município Ano de Fabricação Modelo → https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/F_Frota_por_UF_Municipio_ANO_FAB_MOD_Dezembro_2024.xlsx
  Quantidade de Veículos por UF Município e CEP → https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/A_Frota_por_UF_Municipio_CEP_Dezembro_2024.xlsx
  Quantidade de Veículos por UF Município e Combustível → https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/D_Frota_por_UF_Municipio_COMBUSTIVEL_Dezembro_20241.xlsx
  Quantidade de Veículos por UF Município e Cor → https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/C_F

In [29]:
# Filtrar arquivos "Frota por UF e Tipo de Veículo"
links_uf = [l for l in links if 'FrotaporUF' in l['url'] or 
            'FrotaRegUF' in l['url'] or
            'FrotaReg_UF' in l['url']]

print(f"Arquivos de Frota por UF encontrados: {len(links_uf)}")
for l in links_uf:
    print(f"  {l['texto']} → {l['url'].split('/')[-1]}")

Arquivos de Frota por UF encontrados: 13
  Frota por UF e Tipo de Veículo → FrotaporUFeTipodeveiculoDezembro2024.xlsx
  Frota por UF e Tipo de Veículo → copy_of_FrotaporUFeTipodeVeiculoNovembro20242.xlsx
  Frota por UF e Tipo de Veículo → FrotaporUFeTipodeVeiculoOutubro2024.xlsx
  Frota por UF e Tipo de Veículo → FrotaporUFeTipodeVeculoSetembro2024.xlsx
  Frota por UF e Tipo de Veículo → FrotaporUFeTipodeVeculoAgosto2024.xlsx
  Frota por UF e Tipo de Veículo → FrotaporUFeTipodeVeculoJUL2024.xlsx
  Frota por UF e Tipo de Veículo → copy_of_FrotaporUFeTipodeVeculoJUN2024.xlsx
  Frota por UF e Tipo de Veículo → FrotaporUFeTipodeVeculoMAIO2024.xlsx
  01 - Frota por UF e Tipo de Veículo → FrotaporUFTipoveiculoAbril20241.xlsx
   → FrotaporUFTipodeveiculoMaro2024.xlsx
  Frota por UF e Tipo de Veículo → FrotaporUFeTipodeVeculoMAR2024.xlsx
  Frota por UF e Tipo de Veículo → copy_of_FrotaporUFeTipodeVeculoFEV2024.xlsx
  Frota por UF e Tipo de Veículo → copy2_of_FrotaRegUFTipoModelo06Janeiro2024.x

In [12]:
# Baixar o arquivo de dezembro 2024 para entender a estrutura
url_dez = "https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/FrotaporUFeTipodeveiculoDezembro2024.xlsx"

response_xlsx = requests.get(url_dez)
df_teste = pd.read_excel(BytesIO(response_xlsx.content))

print(f"Shape: {df_teste.shape}")
print(f"\nColunas: {df_teste.columns.tolist()}")
print(f"\nPrimeiras linhas:")
df_teste.head(10)

Shape: (45, 23)

Colunas: ['Frota de veículos, por tipo e com placa, segundo as Grandes Regiões e Unidades da Federação - DEZ/2024', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22']

Primeiras linhas:


,"Frota de veículos, por tipo e com placa, segundo as Grandes Regiões e Unidades da Federação - DEZ/2024",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Grandes Regiões e\nUnidades da Federação,TOTAL,AUTOMÓVEL,BONDE,CAMINHÃO,CAMINHÃO TRATOR,CAMINHONETE,CAMIONETA,CHASSI PLATAFORMA,CICLOMOTOR,...,ÔNIBUS,QUADRICICLO,REBOQUE,SEMI-REBOQUE,SIDE-CAR,OUTROS,TRATOR ESTEIRA,TRATOR RODAS,TRICICLO,UTILITÁRIO
2,Brasil,123974520,63300406,42,3158606,948214,10046145,4571960,1624,522827,...,730316,344,2411954,1386907,8601,39027,236,40551,46211,1800632
3,Norte,7094761,2167696,0,180532,45843,675277,140632,165,20289,...,51914,7,121582,83805,441,1167,3,528,6704,62686
4,Acre,367440,107216,0,8430,1612,35775,5651,10,1107,...,1429,0,5737,2677,60,59,0,3,203,2208
5,Amapá,258190,104084,0,5142,737,29998,5855,14,2309,...,1522,0,1937,2247,50,29,0,9,444,2091
6,Amazonas,1202431,464321,0,23641,5563,113256,31014,37,2220,...,10948,0,5613,16249,7,132,0,120,2568,9440
7,Pará,2810011,765312,0,75715,15876,226197,56188,94,7819,...,22536,3,40209,26496,167,460,1,269,2846,28103
8,Rondônia,1243638,349921,0,33891,9772,135234,18096,8,3319,...,7496,3,24567,17553,40,208,2,55,378,8887
9,Roraima,289075,98232,0,6923,1651,38846,6637,0,1120,...,1585,0,3447,2858,4,49,0,13,90,2511


In [13]:
url_combustivel = "https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/D_Frota_por_UF_Municipio_COMBUSTIVEL_Dezembro_20241.xlsx"

response_comb = requests.get(url_combustivel)
df_comb = pd.read_excel(BytesIO(response_comb.content))

print(f"Shape: {df_comb.shape}")
print(f"\nColunas: {df_comb.columns.tolist()}")
print(f"\nPrimeiras linhas:")
df_comb.head(10)

Shape: (52466, 4)

Colunas: ['UF', 'Município', 'Combustível Veículo', 'Qtd. Veículos']

Primeiras linhas:


,UF,Município,Combustível Veículo,Qtd. Veículos
0,ACRE,ACRELANDIA,ALCOOL,59
1,ACRE,ACRELANDIA,ALCOOL/GASOLINA,2966
2,ACRE,ACRELANDIA,DIESEL,1110
3,ACRE,ACRELANDIA,GASOLINA,3913
4,ACRE,ACRELANDIA,GASOLINA/ALCOOL/GAS NATURAL,2
5,ACRE,ACRELANDIA,Sem Informação,187
6,ACRE,ASSIS BRASIL,ALCOOL,5
7,ACRE,ASSIS BRASIL,ALCOOL/GASOLINA,1122
8,ACRE,ASSIS BRASIL,DIESEL,187
9,ACRE,ASSIS BRASIL,GASOLINA,1044


In [14]:
print(f"Shape: {df_comb.shape}")
print(f"\nColunas: {df_comb.columns.tolist()}")
print(f"\nTipos de combustível únicos:")
print(df_comb.iloc[:, 2].unique())

Shape: (52466, 4)

Colunas: ['UF', 'Município', 'Combustível Veículo', 'Qtd. Veículos']

Tipos de combustível únicos:
['ALCOOL' 'ALCOOL/GASOLINA' 'DIESEL' 'GASOLINA'
 'GASOLINA/ALCOOL/GAS NATURAL' 'Sem Informação'
 'GASOL/GAS NATURAL COMBUSTIVEL' 'GASOLINA/ALCOOL/ELETRICO'
 'GASOLINA/ELETRICO' 'HIBRIDO PLUG-IN' 'ELETRICO/FONTE EXTERNA'
 'GASOLINA/GAS NATURAL VEICULAR' 'ALCOOL/GAS NATURAL COMBUSTIVEL'
 'DIESEL/ELETRICO' 'ELETRICO/FONTE INTERNA' 'HIBRIDO'
 'ALCOOL/GAS NATURAL VEICULAR' 'DIESEL/GAS NATURAL VEICULAR' 'ELETRICO'
 'GAS METANO' 'VIDE/CAMPO/OBSERVACAO' 'GAS NATURAL VEICULAR' 'GASOGENIO'
 'DIESEL/GAS NATURAL COMBUSTIVEL' 'GAS/NATURAL/LIQUEFEITO'
 'Não Identificado' 'Não se Aplica' 'ETANOL/ELETRICO' 'CELULA COMBUSTIVEL']


In [15]:
print(f"Shape: {df_comb.shape}")
print(f"\nColunas: {df_comb.columns.tolist()}")
print(f"\nPrimeiras linhas:")
print(df_comb.head(10))

Shape: (52466, 4)

Colunas: ['UF', 'Município', 'Combustível Veículo', 'Qtd. Veículos']

Primeiras linhas:
     UF     Município          Combustível Veículo  Qtd. Veículos
0  ACRE    ACRELANDIA                       ALCOOL             59
1  ACRE    ACRELANDIA              ALCOOL/GASOLINA           2966
2  ACRE    ACRELANDIA                       DIESEL           1110
3  ACRE    ACRELANDIA                     GASOLINA           3913
4  ACRE    ACRELANDIA  GASOLINA/ALCOOL/GAS NATURAL              2
5  ACRE    ACRELANDIA               Sem Informação            187
6  ACRE  ASSIS BRASIL                       ALCOOL              5
7  ACRE  ASSIS BRASIL              ALCOOL/GASOLINA           1122
8  ACRE  ASSIS BRASIL                       DIESEL            187
9  ACRE  ASSIS BRASIL                     GASOLINA           1044


In [17]:
# Ver todos os tipos de combustível disponíveis
combustiveis = df_comb['Combustível Veículo'].unique()
print(f"Tipos de combustível ({len(combustiveis)}):")
for c in sorted(combustiveis):
    print(f"  {c}")


Tipos de combustível (29):
  ALCOOL
  ALCOOL/GAS NATURAL COMBUSTIVEL
  ALCOOL/GAS NATURAL VEICULAR
  ALCOOL/GASOLINA
  CELULA COMBUSTIVEL
  DIESEL
  DIESEL/ELETRICO
  DIESEL/GAS NATURAL COMBUSTIVEL
  DIESEL/GAS NATURAL VEICULAR
  ELETRICO
  ELETRICO/FONTE EXTERNA
  ELETRICO/FONTE INTERNA
  ETANOL/ELETRICO
  GAS METANO
  GAS NATURAL VEICULAR
  GAS/NATURAL/LIQUEFEITO
  GASOGENIO
  GASOL/GAS NATURAL COMBUSTIVEL
  GASOLINA
  GASOLINA/ALCOOL/ELETRICO
  GASOLINA/ALCOOL/GAS NATURAL
  GASOLINA/ELETRICO
  GASOLINA/GAS NATURAL VEICULAR
  HIBRIDO
  HIBRIDO PLUG-IN
  Não Identificado
  Não se Aplica
  Sem Informação
  VIDE/CAMPO/OBSERVACAO


In [18]:
# Filtrar apenas veículos eletrificados
eletricos = [
    'ELETRICO',
    'ELETRICO/FONTE EXTERNA',
    'ELETRICO/FONTE INTERNA',
    'HIBRIDO',
    'HIBRIDO PLUG-IN',
    'GASOLINA/ELETRICO',
    'ETANOL/ELETRICO',
    'DIESEL/ELETRICO',
    'GASOLINA/ALCOOL/ELETRICO'
]

df_ev = df_comb[df_comb['Combustível Veículo'].isin(eletricos)].copy()

# Agrupar por UF e tipo
df_ev_uf = df_ev.groupby(['UF', 'Combustível Veículo'])['Qtd. Veículos'].sum().reset_index()

print(f"Total de veículos eletrificados (Dez/2024): {df_ev['Qtd. Veículos'].sum():,.0f}")
print(f"\nPor tipo:")
print(df_ev.groupby('Combustível Veículo')['Qtd. Veículos'].sum().sort_values(ascending=False))

Total de veículos eletrificados (Dez/2024): 515,044

Por tipo:
Combustível Veículo
GASOLINA/ELETRICO           164547
ELETRICO/FONTE EXTERNA      140036
GASOLINA/ALCOOL/ELETRICO    113652
ELETRICO/FONTE INTERNA       39444
HIBRIDO PLUG-IN              26919
HIBRIDO                      13302
DIESEL/ELETRICO               9023
ELETRICO                      8119
ETANOL/ELETRICO                  2
Name: Qtd. Veículos, dtype: int64


In [19]:
# Categorizar em 3 grupos
def categorizar_ev(combustivel):
    if combustivel in ['ELETRICO', 'ELETRICO/FONTE EXTERNA', 'ELETRICO/FONTE INTERNA']:
        return 'Elétrico Puro'
    elif combustivel in ['HIBRIDO PLUG-IN', 'DIESEL/ELETRICO', 'ETANOL/ELETRICO']:
        return 'Híbrido Plug-in'
    else:
        return 'Híbrido Convencional'

df_ev['categoria'] = df_ev['Combustível Veículo'].apply(categorizar_ev)

# Resumo por categoria
resumo = df_ev.groupby('categoria')['Qtd. Veículos'].sum().sort_values(ascending=False)
print("Por categoria:")
print(resumo)
print(f"\nTotal: {resumo.sum():,.0f}")

Por categoria:
categoria
Híbrido Convencional    291501
Elétrico Puro           187599
Híbrido Plug-in          35944
Name: Qtd. Veículos, dtype: int64

Total: 515,044


In [20]:
# Verificar quais anos têm o arquivo de combustível disponível
anos = [2021, 2022, 2023, 2024, 2025]

for ano in anos:
    url = f"https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/frota-de-veiculos-{ano}"
    r = requests.get(url)
    soup = BeautifulSoup(r.content, 'html.parser')
    
    links_comb = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if 'COMBUSTIVEL' in href.upper() and '.xls' in href.lower():
            links_comb.append(href.split('/')[-1])
    
    print(f"{ano}: {len(links_comb)} arquivos de combustível")

2021: 12 arquivos de combustível
2022: 12 arquivos de combustível
2023: 12 arquivos de combustível
2024: 12 arquivos de combustível
2025: 12 arquivos de combustível


In [30]:
todos_dados = []
anos = [2021, 2022, 2023, 2024, 2025]

# Mapeamento de mês para número
meses_pt = {
    'Janeiro': 1, 'Fevereiro': 2, 'Março': 3, 'Marco': 3, 'Maro': 3,
    'Abril': 4, 'Maio': 5, 'Junho': 6, 'Julho': 7, 'Agosto': 8,
    'Setembro': 9, 'Outubro': 10, 'Novembro': 11, 'Dezembro': 12,
    'JAN': 1, 'FEV': 2, 'MAR': 3, 'ABR': 4, 'MAI': 5, 'MAI0': 5,
    'JUN': 6, 'JUL': 7, 'AGO': 8, 'SET': 9, 'OUT': 10, 'NOV': 11, 'DEZ': 12
}

def extrair_mes(nome_arquivo):
    nome = nome_arquivo.upper().replace('.XLSX', '').replace('.XLS', '')
    for mes_nome, mes_num in meses_pt.items():
        if mes_nome.upper() in nome:
            return mes_num
    return None

for ano in anos:
    print(f"\nColetando {ano}...")
    url_ano = f"https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/frota-de-veiculos-{ano}"
    r = requests.get(url_ano)
    soup = BeautifulSoup(r.content, 'html.parser')

    links_comb = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if 'COMBUSTIVEL' in href.upper() and '.xls' in href.lower():
            links_comb.append(href)

    for link in links_comb:
        nome_arquivo = link.split('/')[-1]
        mes = extrair_mes(nome_arquivo)

        if mes is None:
            print(f"  Mês não identificado: {nome_arquivo}")
            continue

        try:
            r_xlsx = requests.get(link, timeout=30)
            df_mes = pd.read_excel(BytesIO(r_xlsx.content))

            df_ev = df_mes[df_mes['Combustível Veículo'].isin(eletricos)].copy()

            if len(df_ev) > 0:
                df_ev['ano'] = ano
                df_ev['mes'] = mes
                df_ev['categoria'] = df_ev['Combustível Veículo'].apply(categorizar_ev)
                todos_dados.append(df_ev)
                print(f"  Mês {mes:02d}/{ano}: {df_ev['Qtd. Veículos'].sum():,.0f} veículos eletrificados")

            time.sleep(0.5)

        except Exception as e:
            print(f"  Mês {mes}: Erro — {e}")

print("\nColeta finalizada!")


Coletando 2021...
  Mês 12/2021: 97,920 veículos eletrificados
  Mês 11/2021: 882,850 veículos eletrificados
  Mês 10/2021: 83,992 veículos eletrificados
  Mês 09/2021: 80,852 veículos eletrificados
  Mês 08/2021: 77,629 veículos eletrificados
  Mês 07/2021: 73,268 veículos eletrificados
  Mês 06/2021: 69,748 veículos eletrificados
  Mês 05/2021: 65,414 veículos eletrificados
  Mês 04/2021: 62,343 veículos eletrificados
  Mês 03/2021: 59,782 veículos eletrificados
  Mês 02/2021: 56,988 veículos eletrificados
  Mês 01/2021: 55,631 veículos eletrificados

Coletando 2022...
  Mês 12/2022: 163,181 veículos eletrificados
  Mês 11/2022: 155,847 veículos eletrificados
  Mês 10/2022: 149,273 veículos eletrificados
  Mês 09/2022: 143,362 veículos eletrificados
  Mês 08/2022: 134,070 veículos eletrificados
  Mês 07/2022: 126,944 veículos eletrificados
  Mês 06/2022: 122,448 veículos eletrificados
  Mês 05/2022: 117,731 veículos eletrificados
  Mês 04/2022: 112,552 veículos eletrificados
  Mês 0

In [24]:
# Consolidar todos os dados
df_completo = pd.concat(todos_dados, ignore_index=True)

# Criar coluna de data
df_completo['data'] = pd.to_datetime(
    df_completo['ano'].astype(str) + '-' + 
    df_completo['mes'].astype(str) + '-01'
)

# Agrupar por data e categoria
df_serie = df_completo.groupby(['data', 'ano', 'mes', 'categoria'])['Qtd. Veículos'].sum().reset_index()
df_serie = df_serie.sort_values('data').reset_index(drop=True)

# Ver série temporal total
df_total = df_serie.groupby('data')['Qtd. Veículos'].sum().reset_index()

print("Série temporal — total de eletrificados por mês:")
print(df_total.to_string())

Série temporal — total de eletrificados por mês:
         data  Qtd. Veículos
0  2021-01-01          55631
1  2021-02-01          56988
2  2021-03-01          59782
3  2021-04-01          62343
4  2021-05-01          65414
5  2021-06-01          69748
6  2021-07-01          73268
7  2021-08-01          77629
8  2021-09-01          80852
9  2021-10-01          83992
10 2021-11-01         882850
11 2021-12-01          97920
12 2022-01-01          97920
13 2022-02-01         102516
14 2022-03-01         108560
15 2022-04-01         112552
16 2022-05-01         117731
17 2022-06-01         122448
18 2022-07-01         126944
19 2022-08-01         134070
20 2022-09-01         143362
21 2022-10-01         149273
22 2022-11-01         155847
23 2022-12-01         163181
24 2023-01-01         170053
25 2023-02-01         176572
26 2023-03-01         185745
27 2023-04-01         193593
28 2023-05-01         201900
29 2023-06-01         210280
30 2023-07-01         219457
31 2023-08-01         2

In [25]:
# Tratar outlier de novembro/2021
# Valor esperado = média entre outubro/2021 (83.992) e dezembro/2021 (97.920)
valor_out = df_total.loc[df_total['data'] == '2021-11-01', 'Qtd. Veículos'].values[0]
valor_anterior = df_total.loc[df_total['data'] == '2021-10-01', 'Qtd. Veículos'].values[0]
valor_posterior = df_total.loc[df_total['data'] == '2021-12-01', 'Qtd. Veículos'].values[0]
valor_interpolado = int((valor_anterior + valor_posterior) / 2)

print(f"Outlier detectado em Nov/2021:")
print(f"  Valor original:     {valor_out:,.0f}")
print(f"  Outubro/2021:       {valor_anterior:,.0f}")
print(f"  Dezembro/2021:      {valor_posterior:,.0f}")
print(f"  Valor interpolado:  {valor_interpolado:,.0f}")

# Corrigir na série
df_total.loc[df_total['data'] == '2021-11-01', 'Qtd. Veículos'] = valor_interpolado

print(f"\nSérie corrigida em Nov/2021: {valor_interpolado:,.0f}")

Outlier detectado em Nov/2021:
  Valor original:     882,850
  Outubro/2021:       83,992
  Dezembro/2021:      97,920
  Valor interpolado:  90,956

Série corrigida em Nov/2021: 90,956


In [26]:
# Salvar dado bruto completo
df_completo.to_csv('../data/raw/senatran_eletricos_raw.csv', index=False)

# Salvar série temporal tratada
df_total.to_csv('../output/senatran_serie_temporal.csv', index=False)

# Salvar série por categoria
df_serie_categoria = df_serie.groupby(['data', 'categoria'])['Qtd. Veículos'].sum().reset_index()
df_serie_categoria.to_csv('../output/senatran_serie_categoria.csv', index=False)

print("Arquivos salvos!")
print(f"\nResumo da coleta:")
print(f"  Período: Jan/2021 a Dez/2025")
print(f"  Total de registros brutos: {len(df_completo):,.0f}")
print(f"  Pontos na série temporal: {len(df_total)}")
print(f"  Frota eletrificada Jan/2021: {df_total['Qtd. Veículos'].iloc[0]:,.0f}")
print(f"  Frota eletrificada Dez/2025: {df_total['Qtd. Veículos'].iloc[-1]:,.0f}")
crescimento = ((df_total['Qtd. Veículos'].iloc[-1] / df_total['Qtd. Veículos'].iloc[0]) - 1) * 100
print(f"  Crescimento total: +{crescimento:.0f}%")

Arquivos salvos!

Resumo da coleta:
  Período: Jan/2021 a Dez/2025
  Total de registros brutos: 543,051
  Pontos na série temporal: 60
  Frota eletrificada Jan/2021: 55,631
  Frota eletrificada Dez/2025: 836,151
  Crescimento total: +1403%
